<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/dataset_imbalanced__fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [2]:
import os
import shutil
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())

TensorFlow version: 2.20.0
GPU: /device:GPU:0


In [3]:
BASE            = "/content/newdata"
IMG_SRC         = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR  = "/drive/MyDrive/checkpoints"
MODEL_SAVE_PATH = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/efficientnetv2s_dual_branch.keras"


In [4]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)
shutil.copytree(IMG_SRC, BASE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [5]:
mixed_precision.set_global_policy("float32")

BATCH_SIZE   = 16
IMAGE_SIZE   = 256
FUSION_LAYER = "block4c_add"
EPOCHS       = 30

In [6]:
train_classes = []
class_names = sorted(os.listdir(f"{BASE}/train"))   # alphabetical: melanoma=0, non_melanoma=1
for idx, cls in enumerate(class_names):
    n = len(os.listdir(f"{BASE}/train/{cls}"))
    train_classes.extend([idx] * n)
    print(f"  {cls}: {n} images (class {idx})")

train_labels = np.array(train_classes)
cw = compute_class_weight("balanced", classes=np.unique(train_labels), y=train_labels)
class_weight = dict(enumerate(cw))
print(f"\nClass weights: {class_weight}")
# Expected: melanoma ~4.0, non_melanoma ~0.5

  melanoma: 890 images (class 0)
  non_melanoma: 7122 images (class 1)

Class weights: {0: np.float64(4.501123595505618), 1: np.float64(0.562482448750351)}


In [7]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.1),
], name="data_augmentation")

In [8]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    gray  = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)
    edge  = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge  = edge / (tf.reduce_max(edge) + 1e-6)
    return (image, edge), label

In [9]:
def prepare_dataset(path, shuffle, training=False):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=shuffle,
    )
    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = prepare_dataset(f"{BASE}/train", shuffle=True,  training=True)
val_ds   = prepare_dataset(f"{BASE}/valid", shuffle=False, training=False)
test_ds  = prepare_dataset(f"{BASE}/test",  shuffle=False, training=False)


Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [10]:
def create_dual_model(num_train_batches):
    rgb_input  = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="rgb_input")
    base_model = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    )

    for layer in base_model.layers[:-40]:
        layer.trainable = False

    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=base_model.get_layer(FUSION_LAYER).output,
    )
    middle_feature = feature_extractor(rgb_input)

    # --- Edge branch ---
    edge_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1), name="edge_input")
    x = layers.Conv2D(32,  3, activation="relu", padding="same")(edge_input)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64,  3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.Resizing(middle_feature.shape[1], middle_feature.shape[2])(x)
    x = layers.Conv2D(middle_feature.shape[-1], 1, padding="same")(x)

    # --- Feature fusion ---
    fused = layers.Concatenate()([middle_feature, x])
    fused = layers.Conv2D(256, 3, activation="relu", padding="same")(fused)
    fused = layers.BatchNormalization()(fused)
    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Dense(
        256, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    )(fused)
    fused = layers.Dropout(0.5)(fused)
    outputs = layers.Dense(2, activation="softmax")(fused)

    cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=2e-4,
        decay_steps=EPOCHS * num_train_batches,
    )

    model = tf.keras.Model(inputs=[rgb_input, edge_input], outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cosine_lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),                        # <-- ADDED
            tf.keras.metrics.Recall(name="recall"),                   # <-- ADDED: catches melanoma
            tf.keras.metrics.Precision(name="precision"),             # <-- ADDED
        ],
    )
    return model

In [11]:
model = create_dual_model(num_train_batches=len(train_ds))
model.summary()

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ edge_input          │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        320 │ edge_input[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb_input           │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 16, 16,    │  1,317,880 │ rgb_input[0][0]   │
│ (Functional)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │     16,512 │ resizing[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ functional_1[0][… │
│ (Concatenate)       │ 256)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │    590,080 │ concatenate[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16, 16,    │      1,024 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 2)         │        514 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,084,474 (7.95 MB)

 Trainable params: 766,082 (2.92 MB)

 Non-trainable params: 1,318,392 (5.03 MB)

In [12]:
checkpoint_best = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_dual_{FUSION_LAYER}.keras",
    monitor="val_auc",             # <-- CHANGED: AUC is a better monitor for imbalanced data
    save_best_only=True,
    verbose=1,
)

early_stopping = EarlyStopping(
    monitor="val_auc",             # <-- CHANGED: match checkpoint monitor
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [13]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weight,     # <-- KEY ADDITION: penalises missing melanoma 4x more
    callbacks=[checkpoint_best, early_stopping],
)

Epoch 1/30
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 520ms/step - accuracy: 0.6895 - auc: 0.7819 - loss: 0.5754 - precision: 0.6895 - recall: 0.6895
Epoch 1: val_auc improved from None to 0.74280, saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 324s 582ms/step - accuracy: 0.7214 - auc: 0.8177 - loss: 0.5479 - precision: 0.7214 - recall: 0.7214 - val_accuracy: 0.6224 - val_auc: 0.7428 - val_loss: 0.7303 - val_precision: 0.6224 - val_recall: 0.6224
Epoch 2/30
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - accuracy: 0.7499 - auc: 0.8541 - loss: 0.5038 - precision: 0.7499 - recall: 0.7499
Epoch 2: val_auc improved from 0.74280 to 0.75144, saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 264s 514ms/step - accuracy: 0.7629 - 

In [14]:
results = model.evaluate(test_ds)
print("\n── Test results ──")
for name, val in zip(model.metrics_names, results):
    print(f"  {name}: {val:.4f}")

model.save(MODEL_SAVE_PATH)

63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 371ms/step - accuracy: 0.7345 - auc: 0.8430 - loss: 0.6533 - precision: 0.7345 - recall: 0.7345

── Test results ──
  loss: 0.6533
  compile_metrics: 0.7345
